# `otnsat` DB Setup and Diagnos

0. User specify the DB schema: either `satnrt` or `satdelay` 
1. User provide connection string or .kdbx to `otnsat` Database 
2. Show uploaded deployments and QCed results.
2. Search for SSM results for specified program
3. Test upload SSM resluts to DB.

In [ ]:
from py_nrt.common import get_engine, test_engine_connection
from py_nrt import load_nrt_results as lnr
from py_nrt.sat_qc_result_loader import SatQcResultsLoader
import itables

## User input required

1. General .kdbx can be found in here: https://dalu-my.sharepoint.com/:u:/r/personal/otndc_dal_ca/Documents/otndc/node%20credentials/otnsat_general20260814.kdbx?d=w7060e8f44f7247d19baeb219ab56ff6f&csf=1&web=1&e=uGj73j
2. Set sat_schema to either near real-time (SatQcResultsLoader.SATNRT_SCHEMA) or delay (SatQcResultsLoader.SATDELAY_SCHEMA)

In [ ]:
# engine = get_engine(r"py_nrt/database_conn_string.auth")
engine = get_engine()
test_engine_connection(engine)

sat_schema = SatQcResultsLoader.SATNRT_SCHEMA
# sat_schema = SatQcResultsLoader.SATDELAY_SCHEMA

## Check `otnsat` DB schema connectivity

In [ ]:
sat_loader = SatQcResultsLoader(engine, sat_schema)
sat_loader.check_sat_db()

## If schema does not exist. Please uncomment below code to create the schema and tables.
# sat_loader.init_database_tables()

## Show Deployments by Project

In [ ]:
db_nrt_metadata_df = sat_loader.show_db_deployments()

## Show Alive Tags ( Last Update < 24 hours)

In [ ]:
sat_loader.show_db_nrt_status()

In [ ]:
# Some programs: otn, imos, iraq
programs = sat_loader.get_qced_programs()
ssmoutput_last_modified_map = sat_loader.get_qc_results_for_program(programs[0])
ssmoutput_last_modified_map

## Load SSM output into NRT backend

In [ ]:
summary_df, _ = sat_loader.load_all_qced_results_to_db(['imos'])
if summary_df is not None and not summary_df.empty:
    print('Load summary by tag.')
    itables.show(summary_df)